### Books Recommender system using clustering & collaborative based

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import pickle

Le dataset utilisé pour ce système de recommandation est importé depuis kaggle (ref : https://www.kaggle.com/datasets/ra4u12/bookrecommendation/), il est composé de 3 fichiers de données csv : 
*   La liste des livres "BX-Books" avec leurs informations nécessaires : l'identifiant 'ISBN', le titre du livre 'Book-Title', l'auteur 'Book-Author', la date de publication 'Year-Of-Publication', le publicateur 'Publisher', l'image de couverture du livre en plusieurs taille S,M et L 'Image-URL-S', 'Image-URL-M', 'Image-URL-L' respct
*   La liste des utilisateurs "BX-Users" avec quelques informations : l'identifiant de l'utilisateur 'User-ID', l'adresse 'location', l'age de l'utilisateur 'Age'
*   La liste des notes attribuées par certains utilisateurs "BX-Book-Ratings" aux livres contenant : l'identifiant du livre, l'identifiant de l'utilisateur et la note attribuée.

In [3]:
# Import dataset BX_books 
books = pd.read_csv('data/BX-Books.csv', sep=";", on_bad_lines = "skip", encoding = "latin-1",low_memory=False)

In [4]:
# Explore columns
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [5]:
# data shape
books.shape

(271360, 8)

In [6]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-L']]
# rename columns 
books.rename(columns = {
    "ISBN" : "isbn",
    "Book-Title" : "title",
    "Book-Author" : "author",
    "Year-Of-Publication" : "year",
    "Publisher" : "publisher",
    "Image-URL-L" : "img_url",
}, inplace = True)

In [7]:
books.head()

,isbn,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...


In [8]:
# Import dataset BX_Users
users = pd.read_csv('data/BX-Users.csv', sep=";",on_bad_lines = "skip", encoding = "latin-1",low_memory=False)

In [9]:
# Explore columns
users.columns

Index(['User-ID', 'Location', 'Age'], dtype='object')

In [10]:
# data shape
users.shape

(278858, 3)

In [11]:
# rename columns
users.rename(columns = {
    "User-ID" : "id",
    "Location" : "location",
    "Age" : "age",
}, inplace = True)

In [12]:
users.head()

,id,location,age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [13]:
# Import dataset BX-Book-Ratings
ratings = pd.read_csv('data/BX-Book-Ratings.csv', sep=";", on_bad_lines = "skip", encoding = "latin-1",low_memory=False)

In [14]:
ratings.columns

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')

In [15]:
ratings.rename(columns = {
    "User-ID" : "id",
    "ISBN" : "isbn",
    "Book-Rating" : "rating",
}, inplace = True)

In [16]:
ratings.shape

(1149780, 3)

In [17]:
ratings.head()

,id,isbn,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [18]:
# Keep only the books that have more than 200 ratings from users
x = ratings["id"].value_counts() > 200
x[x].shape

(899,)

In [19]:
y = x[x].index
y

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
       116122,  44296,  28634,  59727,  73681, 274808, 188951,   9856, 155916,
       268622],
      dtype='int64', name='id', length=899)

In [20]:
ratings = ratings[ratings["id"].isin(y)]
ratings.shape

(526356, 3)

In [21]:
# Retrieve the details of each book
books_with_ratings = ratings.merge(books, on = "isbn")
books_with_ratings.head()

,id,isbn,rating,title,author,year,publisher,img_url
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...


In [22]:
num_ratings = books_with_ratings.groupby('title')["rating"].count().reset_index()
num_ratings.head(10)

,title,rating
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1
5,Dark Justice,1
6,Deceived,1
7,Earth Prayers From around the World: 365 Pray...,3
8,Final Fantasy Anthology: Official Strategy Gu...,3
9,Flight of Fancy: American Heiresses (Zebra Ba...,1


In [23]:
num_ratings.rename(columns={'rating':'num_of_rating'}, inplace = True)
num_ratings.head()

,title,num_of_rating
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [24]:
books_with_ratings = books_with_ratings.merge(num_ratings, on = "title")
books_with_ratings.head()

,id,isbn,rating,title,author,year,publisher,img_url,num_of_rating
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,13


In [27]:
data = books_with_ratings[books_with_ratings['num_of_rating']>= 50]

In [28]:
data.drop_duplicates(["id","title"], inplace = True)

C:\Users\Sonia Reffad\AppData\Local\Temp\ipykernel_14632\2745636445.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop_duplicates(["id","title"], inplace = True)


In [29]:
data.shape

(59850, 9)

In [30]:
data.sample(10)

,id,isbn,rating,title,author,year,publisher,img_url,num_of_rating
357436,206074,0312971346,0,High Five (A Stephanie Plum Novel),Janet Evanovich,2000,St. Martin's Paperbacks,http://images.amazon.com/images/P/0312971346.0...,110
456679,256167,0345384369,0,Intensity,DEAN KOONTZ,1996,Ballantine Books,http://images.amazon.com/images/P/0345384369.0...,71
307873,178667,042518630X,0,Purity in Death,J.D. Robb,2002,Berkley Publishing Group,http://images.amazon.com/images/P/042518630X.0...,62
230034,131046,0312983271,7,Full House (Janet Evanovich's Full Series),Janet Evanovich,2002,St. Martin's Paperbacks,http://images.amazon.com/images/P/0312983271.0...,103
222301,128835,0373218400,0,Table For Two,Nora Roberts,2002,Silhouette,http://images.amazon.com/images/P/0373218400.0...,50
262363,153662,0345465083,10,Seabiscuit,LAURA HILLENBRAND,2003,Ballantine Books,http://images.amazon.com/images/P/0345465083.0...,64
393850,227447,0880299185,0,Wuthering Heights,Emily Bronte,1992,Barnes &amp; Noble,http://images.amazon.com/images/P/0880299185.0...,77
348800,201017,0446350982,0,Presumed Innocent,Scott Turow,1988,Warner Books,http://images.amazon.com/images/P/0446350982.0...,149
233254,133747,0842329218,0,Tribulation Force: The Continuing Drama of Tho...,Tim LaHaye,1997,Tyndale House Publishers,http://images.amazon.com/images/P/0842329218.0...,74
270912,156467,0446353957,10,Mirror Image,Sandra Brown,1990,Warner Books,http://images.amazon.com/images/P/0446353957.0...,72


In [31]:
# matrice de contingence
pivot_data = data.pivot_table(columns = 'id', index = 'title', values = 'rating')

In [32]:
pivot_data

id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1st to Die: A Novel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2nd Chance,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4 Blondes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84 Charing Cross Road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,7.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
You Belong To Me,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [33]:
# replace Nan values with 0
pivot_data.fillna(0,inplace=True)

In [34]:
pivot_data

id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
# Matrice sparse
book_sparse = csr_matrix(pivot_data)

In [36]:
# Apply NearestNeighbors algorithm
model = NearestNeighbors(algorithm = 'brute')
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [37]:
# Exemple d'un des livres 
distance, suggestion = model.kneighbors(pivot_data.iloc[237,:].values.reshape(1,-1), n_neighbors = 6)

In [38]:
distance

array([[ 0.        , 67.75691847, 68.05145112, 72.277244  , 75.81556568,
        76.30203143]])

In [39]:
for i in range(len(suggestion)) :
    print(pivot_data.index[suggestion[i]])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'The Cradle Will Fall'],
      dtype='object', name='title')


In [40]:
books_name = pivot_data.index

In [41]:
# save results on files
pickle.dump(model, open('artifacts/model.pkl','wb'))
pickle.dump(books_name, open('artifacts/books_name.pkl','wb'))
pickle.dump(data, open('artifacts/data.pkl','wb'))
pickle.dump(pivot_data, open('artifacts/pivot_data.pkl','wb'))

In [42]:
def recommend_book(book_name):
    book_id = np.where(pivot_data.index ==book_name)[0][0]
    distance, suggestion = model.kneighbors(pivot_data.iloc[book_id,:].values.reshape(1,-1), n_neighbors = 6)
    books = pivot_data.index[suggestion[0]]
    for j in books : 
        print(j)

In [47]:
recommend_book('A Civil Action')

A Civil Action
No Safe Place
Exclusive
Jacob Have I Loved
Long After Midnight
The Cradle Will Fall
